# Contrastive Image–Text Model on Synthetic Shapes

A compact, production-styled implementation of a **dual-encoder contrastive image–text model**
(the CLIP training recipe) on a fully synthetic dataset of coloured geometric shapes.

The notebook is organised as three explicit **pipelines** so each stage can be run, tested, or
swapped independently:

| Pipeline | Responsibility |
|---|---|
| `DataPipeline` | render images + captions, tokenise, build loaders |
| `TrainingPipeline` | fit the two encoders with a symmetric contrastive loss |
| `RetrievalPipeline` | encode a query and return the closest items from a bank |

Everything here is written from scratch; the only borrowed element is the *idea* of aligning
image and text embeddings in a shared space, a standard publicly documented technique.


## Imports

In [ ]:
from __future__ import annotations

import logging
import random
from dataclasses import dataclass, field
from typing import Sequence

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw
from torch.utils.data import DataLoader, Dataset

import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger("contrastive_shapes")

## Configuration

A single dataclass holds every tunable knob. Edit it here rather than hunting through the code.

In [ ]:
@dataclass
class Config:
    """Single source of truth for every tunable knob."""

    # data
    image_size: int = 32
    colors: Sequence[str] = field(
        default_factory=lambda: [
            "red", "green", "blue", "yellow", "purple",
            "orange", "pink", "brown", "gray",
        ]
    )
    shapes: Sequence[str] = field(default_factory=lambda: ["square", "circle", "triangle"])
    positions: Sequence[str] = field(
        default_factory=lambda: [
            "left", "center", "right", "top", "bottom",
            "top-left", "top-right", "bottom-left", "bottom-right",
        ]
    )
    val_fraction: float = 0.2

    # model
    embed_dim: int = 32
    attn_heads: int = 4
    max_tokens: int = 4  # [CLS] + color + shape + position

    # optimisation
    batch_size: int = 32
    epochs: int = 60
    lr: float = 3e-4
    weight_decay: float = 1e-2
    init_temperature: float = 0.07  # only the *initial* value; it is learned

    # misc
    seed: int = 0
    device: str = field(default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu")


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

## Rendering

Each position is decomposed into an **(x-band, y-band)** pair, where a band is a `(start, end)`
fraction of the drawable area. This explicit lookup replaces brittle substring checks.


In [ ]:
_X_BANDS = {"left": (0.00, 0.50), "right": (0.50, 1.00), "center": (0.25, 0.75)}
_Y_BANDS = {"top": (0.00, 0.50), "bottom": (0.50, 1.00), "center": (0.25, 0.75)}


def _resolve_bands(position: str) -> tuple[tuple[float, float], tuple[float, float]]:
    """Map a position name to (x_band, y_band)."""
    parts = position.split("-")  # e.g. "top-right" -> ["top", "right"]
    x_key = next((p for p in parts if p in _X_BANDS), "center")
    y_key = next((p for p in parts if p in _Y_BANDS), "center")
    return _X_BANDS[x_key], _Y_BANDS[y_key]


class ShapeRenderer:
    """Draws a single coloured shape at a named position on a white canvas."""

    def __init__(self, image_size: int) -> None:
        self.size = image_size
        self.margin = max(2, image_size // 6)

    def render(self, color: str, shape: str, position: str) -> Image.Image:
        canvas = Image.new("RGB", (self.size, self.size), "white")
        draw = ImageDraw.Draw(canvas)

        span = self.size - 2 * self.margin
        (xa, xb), (ya, yb) = _resolve_bands(position)
        x0 = self.margin + int(xa * span)
        x1 = self.margin + int(xb * span)
        y0 = self.margin + int(ya * span)
        y1 = self.margin + int(yb * span)

        box = [x0, y0, x1, y1]
        if shape == "square":
            draw.rectangle(box, fill=color, outline="black")
        elif shape == "circle":
            draw.ellipse(box, fill=color, outline="black")
        elif shape == "triangle":
            draw.polygon([((x0 + x1) // 2, y0), (x0, y1), (x1, y1)], fill=color, outline="black")
        else:
            raise ValueError(f"unknown shape: {shape!r}")
        return canvas


def image_to_tensor(img: Image.Image) -> torch.Tensor:
    """PIL RGB image -> CxHxW float tensor in [0, 1]."""
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1).contiguous()

Quick visual check that the renderer places shapes where the caption says:

In [ ]:
_r = ShapeRenderer(64)
_examples = [("red", "circle", "top-right"), ("blue", "square", "center"), ("green", "triangle", "bottom-left")]
fig, axes = plt.subplots(1, len(_examples), figsize=(6, 2))
for ax, (c, s, p) in zip(axes, _examples):
    ax.imshow(_r.render(c, s, p)); ax.set_title(f"{c} {s}\n{p}", fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## Tokenisation

A minimal whitespace tokenizer with a fixed vocabulary, plus a light `normalise()` layer so that
loosely phrased queries (`"circle on right top"`) still map onto the tokens the model was trained on.


In [ ]:
class Tokenizer:
    """Whitespace tokenizer with a fixed vocab and a query-normalisation layer."""

    CLS = "[CLS]"
    _ALIASES = {
        "right top": "top-right", "top right": "top-right",
        "left top": "top-left", "top left": "top-left",
        "right bottom": "bottom-right", "bottom right": "bottom-right",
        "left bottom": "bottom-left", "bottom left": "bottom-left",
        "middle": "center", "centre": "center",
    }
    _STOP = {"a", "an", "the", "on", "in", "at", "of", "is"}

    def __init__(self, captions: Sequence[str]) -> None:
        vocab = sorted({w for cap in captions for w in cap.split()})
        self.itos = [self.CLS, *vocab]
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}

    def __len__(self) -> int:
        return len(self.itos)

    def normalise(self, text: str) -> str:
        """Best-effort clean-up of free-form text into canonical tokens."""
        text = text.lower().strip()
        for phrase, canonical in self._ALIASES.items():
            text = text.replace(phrase, canonical)
        kept = [w for w in text.split() if w not in self._STOP]
        return " ".join(kept)

    def encode(self, text: str, *, normalise: bool = False) -> torch.Tensor:
        if normalise:
            text = self.normalise(text)
        ids = [self.stoi[self.CLS]]
        for word in text.split():
            if word not in self.stoi:
                raise KeyError(f"token {word!r} is not in the vocabulary {self.itos}")
            ids.append(self.stoi[word])
        return torch.tensor(ids, dtype=torch.long)

## Dataset & DataPipeline

Renders the full product space of `(color, shape, position)` and serves seeded train/val loaders.

In [ ]:
class ShapeCaptionDataset(Dataset):
    """Holds pre-rendered image tensors alongside their tokenised captions."""

    def __init__(self, images: torch.Tensor, token_ids: torch.Tensor, captions: list[str]) -> None:
        self.images = images
        self.token_ids = token_ids
        self.captions = captions

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, idx: int):
        return self.images[idx], self.token_ids[idx], self.captions[idx]


class DataPipeline:
    """Renders the full product space of (color, shape, position) and serves loaders."""

    def __init__(self, cfg: Config) -> None:
        self.cfg = cfg
        self.renderer = ShapeRenderer(cfg.image_size)

        captions, images = [], []
        for color in cfg.colors:
            for shape in cfg.shapes:
                for position in cfg.positions:
                    images.append(image_to_tensor(self.renderer.render(color, shape, position)))
                    captions.append(f"{color} {shape} {position}")

        self.captions = captions
        self.images = torch.stack(images)
        self.tokenizer = Tokenizer(captions)
        self.token_ids = torch.stack([self.tokenizer.encode(c) for c in captions])

        log.info("rendered %d image-caption pairs | vocab=%d", len(captions), len(self.tokenizer))

    def build_loaders(self) -> tuple[DataLoader, DataLoader]:
        n = len(self.captions)
        idx = torch.randperm(n, generator=torch.Generator().manual_seed(self.cfg.seed))
        n_val = int(self.cfg.val_fraction * n)
        val_idx, train_idx = idx[:n_val], idx[n_val:]

        def make(indices: torch.Tensor, shuffle: bool) -> DataLoader:
            subset = ShapeCaptionDataset(
                self.images[indices],
                self.token_ids[indices],
                [self.captions[i] for i in indices.tolist()],
            )
            return DataLoader(subset, batch_size=self.cfg.batch_size, shuffle=shuffle)

        return make(train_idx, True), make(val_idx, False)

## Model

Two towers project into one shared space:

- **`VisionEncoder`** — a small conv stack, global-average-pooled and projected.
- **`TextEncoder`** — token + positional embeddings, one self-attention block, `[CLS]` readout.
- **`DualEncoderCLIP`** — wraps both and holds a **learned temperature** (`logit_scale`).

Training uses a **symmetric InfoNCE** loss: each image must match its caption and vice versa.


In [ ]:
class VisionEncoder(nn.Module):
    """Small conv stack -> global-average-pool -> projection to the shared space."""

    def __init__(self, embed_dim: int) -> None:
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.GELU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.GELU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.GELU(),
        )
        self.head = nn.Sequential(nn.Linear(256, embed_dim), nn.LayerNorm(embed_dim))

    def forward(self, pixels: torch.Tensor) -> torch.Tensor:
        feats = self.backbone(pixels).mean(dim=(2, 3))  # global average pool
        return F.normalize(self.head(feats), dim=-1)


class TextEncoder(nn.Module):
    """Token + positional embeddings -> self-attention -> [CLS] readout."""

    def __init__(self, vocab_size: int, embed_dim: int, heads: int, max_tokens: int) -> None:
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, embed_dim)
        self.pos_emb = nn.Embedding(max_tokens, embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, heads, batch_first=True)
        self.head = nn.Sequential(nn.Linear(embed_dim, embed_dim), nn.LayerNorm(embed_dim))

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        _, length = token_ids.shape
        positions = torch.arange(length, device=token_ids.device)
        x = self.token_emb(token_ids) + self.pos_emb(positions)
        x, _ = self.attn(x, x, x)
        cls = x[:, 0]  # readout from the [CLS] slot
        return F.normalize(self.head(cls), dim=-1)


class DualEncoderCLIP(nn.Module):
    """Two towers projecting into one space, with a learned temperature."""

    def __init__(self, cfg: Config, vocab_size: int) -> None:
        super().__init__()
        self.vision = VisionEncoder(cfg.embed_dim)
        self.text = TextEncoder(vocab_size, cfg.embed_dim, cfg.attn_heads, cfg.max_tokens)
        self.logit_scale = nn.Parameter(
            torch.tensor(np.log(1.0 / cfg.init_temperature), dtype=torch.float32)
        )

    def encode_image(self, pixels: torch.Tensor) -> torch.Tensor:
        return self.vision(pixels)

    def encode_text(self, token_ids: torch.Tensor) -> torch.Tensor:
        return self.text(token_ids)

    def forward(self, pixels, token_ids):
        img = self.encode_image(pixels)
        txt = self.encode_text(token_ids)
        scale = self.logit_scale.clamp(max=np.log(100.0)).exp()
        return img, txt, scale


def contrastive_loss(img: torch.Tensor, txt: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
    """Symmetric InfoNCE: match each image to its caption and vice versa."""
    logits = scale * img @ txt.t()
    target = torch.arange(img.size(0), device=img.device)
    return 0.5 * (F.cross_entropy(logits, target) + F.cross_entropy(logits.t(), target))

## TrainingPipeline

The training loop with the standard stabilisers: **gradient clipping**, a **cosine LR schedule**,
loss normalised by true sample count, and **best-on-validation checkpoint restore**.


In [ ]:
class TrainingPipeline:
    def __init__(self, model: DualEncoderCLIP, cfg: Config) -> None:
        self.model = model.to(cfg.device)
        self.cfg = cfg
        self.opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        self.sched = torch.optim.lr_scheduler.CosineAnnealingLR(self.opt, T_max=cfg.epochs)

    def _run_epoch(self, loader: DataLoader, train: bool) -> float:
        self.model.train(train)
        total, count = 0.0, 0
        torch.set_grad_enabled(train)
        for pixels, token_ids, _ in loader:
            pixels, token_ids = pixels.to(self.cfg.device), token_ids.to(self.cfg.device)
            img, txt, scale = self.model(pixels, token_ids)
            loss = contrastive_loss(img, txt, scale)
            if train:
                self.opt.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                self.opt.step()
            total += loss.item() * pixels.size(0)
            count += pixels.size(0)
        torch.set_grad_enabled(True)
        return total / count  # normalised by true sample count, not batch count

    def fit(self, train_loader: DataLoader, val_loader: DataLoader) -> dict[str, float]:
        best_val = float("inf")
        best_state = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
        for epoch in range(1, self.cfg.epochs + 1):
            train_loss = self._run_epoch(train_loader, train=True)
            val_loss = self._run_epoch(val_loader, train=False)
            self.sched.step()
            if val_loss < best_val:
                best_val = val_loss
                best_state = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
            if epoch == 1 or epoch % 5 == 0 or epoch == self.cfg.epochs:
                log.info("epoch %02d | train %.4f | val %.4f", epoch, train_loss, val_loss)
        self.model.load_state_dict(best_state)  # restore best-on-val weights
        log.info("restored best checkpoint | val %.4f", best_val)
        return {"best_val_loss": best_val}

## RetrievalPipeline

Wraps a trained model to answer **text→image** and **image→text** queries. Call `.index()` once to
pre-encode a gallery, after which every query is a single matrix multiply.


In [ ]:
class RetrievalPipeline:
    """Wraps a trained model to answer text->image and image->text queries."""

    def __init__(self, model: DualEncoderCLIP, tokenizer: Tokenizer, cfg: Config) -> None:
        self.model = model.eval().to(cfg.device)
        self.tokenizer = tokenizer
        self.cfg = cfg
        self._image_bank = None
        self._caption_bank = None
        self._captions: list[str] = []
        self._images = None

    @torch.no_grad()
    def index(self, images: torch.Tensor, captions: list[str]) -> "RetrievalPipeline":
        """Pre-encode a gallery so queries are a single matrix multiply."""
        self._images = images
        self._captions = captions
        self._image_bank = self.model.encode_image(images.to(self.cfg.device))
        token_ids = torch.stack([self.tokenizer.encode(c) for c in captions]).to(self.cfg.device)
        self._caption_bank = self.model.encode_text(token_ids)
        return self

    @torch.no_grad()
    def text_to_image(self, query: str, k: int = 5) -> list[tuple[str, float]]:
        """Return the captions of the top-k gallery images closest to `query`."""
        token_ids = self.tokenizer.encode(query, normalise=True).unsqueeze(0).to(self.cfg.device)
        q = self.model.encode_text(token_ids)
        scores = (q @ self._image_bank.t()).softmax(dim=1).squeeze(0)
        top = scores.topk(min(k, scores.numel())).indices.tolist()
        return [(self._captions[i], float(scores[i])) for i in top]

    @torch.no_grad()
    def image_to_text(self, image_idx: int, k: int = 5) -> list[tuple[str, float]]:
        """Return the top-k captions for gallery image `image_idx`."""
        q = self._image_bank[image_idx : image_idx + 1]
        scores = (q @ self._caption_bank.t()).softmax(dim=1).squeeze(0)
        top = scores.topk(min(k, scores.numel())).indices.tolist()
        return [(self._captions[i], float(scores[i])) for i in top]

    @torch.no_grad()
    def top1_accuracy(self) -> float:
        """Diagonal retrieval accuracy over the indexed gallery."""
        sims = self._image_bank @ self._caption_bank.t()
        pred = sims.argmax(dim=1)
        target = torch.arange(len(self._captions), device=pred.device)
        return float((pred == target).float().mean())

## Train

Instantiate the config, build the data, and fit the model.

In [ ]:
cfg = Config(epochs=60)          # tweak knobs here
log.info("device=%s", cfg.device)

seed_everything(cfg.seed)
data = DataPipeline(cfg)
train_loader, val_loader = data.build_loaders()

model = DualEncoderCLIP(cfg, vocab_size=len(data.tokenizer))
TrainingPipeline(model, cfg).fit(train_loader, val_loader)

## Evaluate & query

Index the full gallery, report retrieval accuracy, and run a few text→image queries.

In [ ]:
engine = RetrievalPipeline(model, data.tokenizer, cfg).index(data.images, data.captions)
log.info("gallery top-1 retrieval accuracy: %.3f", engine.top1_accuracy())

for query in ["red circle top-right", "circle on right top", "blue square middle"]:
    try:
        results = engine.text_to_image(query, k=3)
    except KeyError as err:
        log.warning("query %r rejected: %s", query, err)
        continue
    pretty = ", ".join(f"{cap} ({score:.2f})" for cap, score in results)
    log.info("text->image | %-24s -> %s", query, pretty)

Visualise a text→image query — the top retrieved gallery images for a phrase:

In [ ]:
def show_text_query(query: str, k: int = 4):
    results = engine.text_to_image(query, k=k)
    name_to_idx = {c: i for i, c in enumerate(data.captions)}
    fig, axes = plt.subplots(1, k, figsize=(2 * k, 2.2))
    for ax, (cap, score) in zip(axes, results):
        img = data.images[name_to_idx[cap]].permute(1, 2, 0).numpy()
        ax.imshow(img); ax.set_title(f"{cap}\n{score:.2f}", fontsize=7); ax.axis("off")
    fig.suptitle(f"query: '{query}'", fontsize=9); plt.tight_layout(); plt.show()

show_text_query("red circle top-right")
show_text_query("blue square middle")